# 异常检测

在本练习中，你将实现异常检测算法，并将其用于检测网络中出故障的服务器。



# 目录
- [ 1 - 程序包 ](#1)
- [ 2 - 异常检测](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2  数据集](#2.2)
  - [ 2.3 高斯分布](#2.3)
    - [ 练习 1](#ex01)
    - [ 练习 2](#ex02)
  - [ 2.4 高维数据集](#2.4)


<a name="1"></a>
## 1 - 程序包

首先，运行下面的单元格，导入本次作业中会用到的所有程序包。
- [numpy](www.numpy.org) 是 Python 中处理矩阵的基础程序包。
- [matplotlib](http://matplotlib.org) 是一个著名的 Python 绘图库。
- ``utils.py`` 包含本次作业的辅助函数。你不需要修改该文件中的代码。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *

%matplotlib inline

<a name="2"></a>
## 2 - 异常检测

<a name="2.1"></a>
### 2.1 问题陈述

在本练习中，你将实现一个异常检测算法，用于检测服务器计算机中的异常行为。

数据集包含两个特征——
   * 吞吐量（mb/s）和
   * 每台服务器响应的延迟（ms）。

在服务器运行期间，你收集了 $m=307$ 个样本，记录它们的行为，因此得到无标签数据集 $\{x^{(1)}, \ldots, x^{(m)}\}$。
* 你怀疑这些样本中的绝大多数是服务器正常运行的“正常”（非异常）样本，但该数据集中也可能存在一些服务器行为异常的样本。

你将用高斯模型来检测数据集中的异常样本。
* 你将先从二维数据集入手，以便可视化算法在做什么。
* 在该数据集上，你将拟合一个高斯分布，然后找出概率很低、因而可视为异常的取值。
* 之后，你会把异常检测算法应用于维度更多的大型数据集。

<a name="2.2"></a>
### 2.2  数据集

你将先加载本次任务的数据集。
- 下面显示的 `load_data()` 函数把数据加载到变量 `X_train`、`X_val` 和 `y_val` 中
    - 你将用 `X_train` 拟合高斯分布
    - 你将把 `X_val` 和 `y_val` 用作交叉验证集，以选取阈值并判定样本是异常还是正常

In [ ]:
# 加载数据集
X_train, X_val, y_val = load_data()

#### 查看变量
让我们更熟悉你的数据集。
- 一个好的起点是直接把每个变量打印出来，看看它包含什么。

下面的代码会打印每个变量的前五个元素

In [ ]:
# 显示 X_train 的前五个元素
print("The first 5 elements of X_train are:\n", X_train[:5])  

In [ ]:
# 显示 X_val 的前五个元素
print("The first 5 elements of X_val are\n", X_val[:5])  

In [ ]:
# 显示 y_val 的前五个元素
print("The first 5 elements of y_val are\n", y_val[:5])  

#### 查看变量的维度

熟悉数据的另一个有用方式是查看其维度。

下面的代码会打印 `X_train`、`X_val` 和 `y_val` 的形状。

In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of X_val is:', X_val.shape)
print ('The shape of y_val is: ', y_val.shape)

#### 可视化数据

在开始任何任务之前，通过可视化来理解数据往往很有用。
- 对这个数据集，你可以用散点图可视化数据（`X_train`），因为它只有两个可绘制的属性（吞吐量和延迟）

- 你的图应与下图相似
<img src="图片/figure1.png" width="500" height="500">

> **图中文字（中文）**：横轴：延迟（Latency，ms）；纵轴：吞吐量 图 1：第一个数据集。

In [ ]:
# 创建数据的散点图。若要把标记改成蓝色的 "x"，
# 我们使用了 'marker' 和 'c' 参数
plt.scatter(X_train[:, 0], X_train[:, 1], marker='x', c='b') 

# 设置标题
plt.title("The first dataset")
# 设置 y 轴标签
plt.ylabel('Throughput (mb/s)')
# 设置 x 轴标签
plt.xlabel('Latency (ms)')
# 设置坐标轴范围
plt.axis([0, 30, 0, 30])
plt.show()

<a name="2.3"></a>
### 2.3 高斯分布

要进行异常检测，你首先需要为数据的分布拟合一个模型。

* 给定训练集 $\{x^{(1)}, ..., x^{(m)}\}$，你希望估计每个特征 $x_i$ 的高斯分布。

* 回忆一下，高斯分布由下式给出

   $$ p(x ; \mu,\sigma ^2) = \frac{1}{\sqrt{2 \pi \sigma ^2}}\exp^{ - \frac{(x - \mu)^2}{2 \sigma ^2} }$$

   其中 $\mu$ 是均值，$\sigma^2$ 控制方差。
   
* 对每个特征 $i = 1\ldots n$，你需要找到拟合第 $i$ 维数据 $\{x_i^{(1)}, ..., x_i^{(m)}\}$（每个样本的第 $i$ 维）的参数 $\mu_i$ 和 $\sigma_i^2$。

### 2.2.1 估计高斯分布的参数

**实现**：

你的任务是补全下面 `estimate_gaussian` 中的代码。

<a name="ex01"></a>
### 练习 1

请补全下面的 `estimate_gaussian` 函数，计算 `mu`（`X` 中每个特征的均值）和 `var`（`X` 中每个特征的方差）。

你可以用下面的公式估计第 $i$ 个特征的参数（$\mu_i$, $\sigma_i^2$）。估计均值时，你使用：

$$\mu_i = \frac{1}{m} \sum_{j=1}^m x_i^{(j)}$$

估计方差时，你使用：
$$\sigma_i^2 = \frac{1}{m} \sum_{j=1}^m (x_i^{(j)} - \mu_i)^2$$

如果你卡住了，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [ ]:
# UNQ_C1
# 计分函数：estimate_gaussian

def estimate_gaussian(X): 
    """计算数据集中所有特征的均值和方差
   
    参数：
        X (ndarray): (m, n) 数据矩阵
   
    返回：
        mu (ndarray): (n,) 所有特征的均值
        var (ndarray): (n,) 所有特征的方差"""

    m, n = X.shape
    
    ### 在此处编写代码 ###
    
    ### 代码编写结束 ###
        
    return mu, var

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

* 你可以用两种方式实现该函数：
  * 1 - 用两层嵌套的 for 循环——一层遍历 <code>X</code> 的**列**（每个特征），再遍历每个数据点。
  * 2 - 用向量化方式，借助带 <code>axis = 0</code> 参数的 <code>np.sum()</code>（因为我们要对每一列求和）


* 该函数向量化实现的整体结构可以这样写：

```python  
def estimate_gaussian(X): 
    m, n = X.shape

    ### START CODE HERE ### 
    mu = # Your code here to calculate the mean of every feature
    var = # Your code here to calculate the variance of every feature 
    ### END CODE HERE ### 

    return mu, var
```


如果还是卡住，可以查看下面给出的提示，弄清如何计算 <code>mu</code> 和 <code>var</code>。

<details>
      <summary><font size="2" color="darkblue"><b>计算 mu 的提示</b></font></summary>
       &emsp; &emsp; 你可以用带 <code>axis = 0</code> 参数的 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> 来求数组每一列的和
      <details>
          <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 mu 的更多提示</b></font></summary>
           &emsp; &emsp; 你可以这样计算 mu：<code>mu = 1 / m * np.sum(X, axis = 0)</code>
       </details>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 var 的提示</b></font></summary>
       &emsp; &emsp; 你可以用带 <code>axis = 0</code> 参数的 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> 来求数组每一列的和，再用 <code>**2</code> 求平方。
      <details>
          <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 var 的更多提示</b></font></summary>
           &emsp; &emsp; 你可以这样计算 var：<code> var = 1 / m * np.sum((X - mu) ** 2, axis = 0)</code>
       </details>
</details>

</details>

你可以运行下面的测试代码，检查你的实现是否正确：

In [ ]:
# 估计每个特征的均值和方差
mu, var = estimate_gaussian(X_train)              

print("Mean of each feature:", mu)
print("Variance of each feature:", var)
    
# 单元测试
from public_tests import *
estimate_gaussian_test(estimate_gaussian)

**预期输出**：
<table>
  <tr>
    <td> <b>每个特征的均值：<b>  </td> 
    <td> [14.11222578 14.99771051]</td> 
   </tr>    
   <tr>
    <td> <b>每个特征的方差：<b>  </td>
     <td> [1.83263141 1.70974533] </td> 
  </tr>
</table>

既然你已经完成 `estimate_gaussian` 中的代码，我们将可视化拟合出的高斯分布的等高线。

你应得到与下图相似的图。
<img src="图片/figure2.png" width="500" height="500">

> **图中文字（中文）**：横轴：延迟（Latency，ms） 图 2：拟合到该数据集的分布的高斯等高线。


从你的图中可以看出，大多数样本都位于概率最高的区域，而异常样本则位于概率较低的区域。

In [ ]:
# 返回多元正态分布的密度
# 在 X_train 的每个数据点（行）处
p = multivariate_gaussian(X_train, mu, var)

#绘图代码
visualize_fit(X_train, mu, var)

### 2.2.2 选取阈值 $\epsilon$

既然你已经估计出高斯参数，就可以考察在这个分布下哪些样本的概率很高、哪些样本的概率很低。

* 概率低的样本更可能是我们数据集中的异常。
* 判定哪些样本是异常的一种方法是：基于交叉验证集选取一个阈值。

在本节中，你将补全 `select_threshold` 中的代码，用交叉验证集上的 $F_1$ 分数来选取阈值 $\varepsilon$。

* 为此，我们将使用交叉验证集
$\{(x_{\rm cv}^{(1)}, y_{\rm cv}^{(1)}),\ldots, (x_{\rm cv}^{(m_{\rm cv})}, y_{\rm cv}^{(m_{\rm cv})})\}$，其中标签 $y=1$ 对应异常样本，$y=0$ 对应正常样本。
* 对每个交叉验证样本，我们将计算 $p(x_{\rm cv}^{(i)})$。所有这些概率构成的向量 $p(x_{\rm cv}^{(1)}), \ldots, p(x_{\rm cv}^{(m_{\rm cv)}})$ 通过向量 `p_val` 传给 `select_threshold`。
* 对应的标签 $y_{\rm cv}^{(1)}, \ldots, y_{\rm cv}^{(m_{\rm cv)}}$ 通过向量 `y_val` 传给同一个函数。

<a name="ex02"></a>
### 练习 2
请补全下面的 `select_threshold` 函数，根据验证集结果（`p_val`）和真实标签（`y_val`）找出用于判定离群点的最佳阈值。

* 在给定的 `select_threshold` 代码中，已经有一个循环会尝试许多不同的 $\varepsilon$ 取值，并根据 $F_1$ 分数选出最佳的 $\varepsilon$。

* 你需要实现代码，计算以 `epsilon` 作为阈值时的 F1 分数，并把该值放入 `F1`。

  * 回忆一下，若样本 $x$ 的概率很低，即 $p(x) < \varepsilon$，则它被判定为异常。
        
  * 然后，你可以用下式计算精确率（precision）和召回率（recall）：
   $$\begin{aligned}
   prec&=&\frac{tp}{tp+fp}\\
   rec&=&\frac{tp}{tp+fn},
   \end{aligned}$$ 其中
    * $tp$ 是真正例的数量：真实标签说它是异常，我们的算法也正确地把它判定为异常。
    * $fp$ 是假正例的数量：真实标签说它不是异常，但我们的算法错误地把它判定为异常。
    * $fn$ 是假负例的数量：真实标签说它是异常，但我们的算法错误地把它判定为非异常。

  * $F_1$ 分数用精确率（$prec$）和召回率（$rec$）计算如下：
    $$F_1 = \frac{2\cdot prec \cdot rec}{prec + rec}$$ 

**实现提示：**
为计算 $tp$、$fp$ 和 $fn$，你可以采用向量化实现，而不必遍历所有样本。


如果你卡住了，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [ ]:
# UNQ_C2
# 计分函数：select_threshold

def select_threshold(y_val, p_val): 
    """根据验证集上的结果（p_val）和真实标签（y_val），
   找出用于选出离群点的最佳阈值
   
    参数：
        y_val (ndarray): 验证集上的真实标签
        p_val (ndarray): 验证集上的结果
       
    返回：
        epsilon (float): 选出的阈值
        F1 (float):      以 epsilon 为阈值时的 F1 分数""" 

    best_epsilon = 0
    best_F1 = 0
    F1 = 0
    
    step_size = (max(p_val) - min(p_val)) / 1000
    
    for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
        ### 在此处编写代码 ###
        
        ### 代码编写结束 ###
        
        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon
        
    return best_epsilon, best_F1

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

* 该函数向量化实现的整体结构可以这样写：

```python  
def select_threshold(y_val, p_val): 
    best_epsilon = 0
    best_F1 = 0
    F1 = 0

    step_size = (max(p_val) - min(p_val)) / 1000

    for epsilon in np.arange(min(p_val), max(p_val), step_size):

        ### START CODE HERE ### 
        predictions = # Your code here to calculate predictions for each example using epsilon as threshold

        tp = # Your code here to calculate number of true positives
        fp = # Your code here to calculate number of false positives
        fn = # Your code here to calculate number of false negatives

        prec = # Your code here to calculate precision
        rec = # Your code here to calculate recall

        F1 = # Your code here to calculate F1
        ### END CODE HERE ### 

        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon

    return best_epsilon, best_F1
```


如果还是卡住，可以查看下面给出的提示，弄清如何计算各个变量。

<details>
      <summary><font size="2" color="darkblue"><b>计算 predictions 的提示</b></font></summary>
       &emsp; &emsp; 若样本 𝑥 的概率很低，即 $p(x) < \epsilon$，则它被判定为异常。要为每个样本得到预测结果（0/False 表示正常，1/True 表示异常），你可以用 <code>predictions = (p_val < epsilon)</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 tp、fp、fn 的提示</b></font></summary>
       &emsp; &emsp; 
    <ul>
      <li>若 $n$ 维二值向量中有若干个二值取值，你可以用 <code>np.sum(v == 0)</code> 找出该向量中有多少个值为 0</li>
      <li>你也可以对这类二值向量施加逻辑*与*运算。例如，<code>predictions</code> 是一个长度等于交叉验证集样本数的二值向量，若算法认为 $x_{\rm cv}^{(i)}$ 是异常，则第 $i$ 个元素为 1，否则为 0。</li>
      <li>于是，例如你可以用下式计算假正例的数量：
<code>fp = sum((predictions == 1) & (y_val == 0))</code>。</li>
    </ul>
     <details>
          <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 tp、fn 的更多提示</b></font></summary>
           &emsp; &emsp;
         <ul>
          <li>你可以这样计算 tp：<code> tp = np.sum((predictions == 1) & (y_val == 1))</code></li>
          <li>你可以这样计算 tn：<code> fn = np.sum((predictions == 0) & (y_val == 1))</code></li>  
          </ul>
      </details>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算精确率的提示</b></font></summary>
       &emsp; &emsp; 你可以这样计算精确率：<code>prec = tp / (tp + fp)</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算召回率的提示</b></font></summary>
       &emsp; &emsp; 你可以这样计算召回率：<code>rec = tp / (tp + fn)</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 F1 的提示</b></font></summary>
       &emsp; &emsp; 你可以这样计算 F1：<code>F1 = 2 * prec * rec / (prec + rec)</code>
</details>

</details>

你可以用下面的代码检查你的实现

In [ ]:
p_val = multivariate_gaussian(X_val, mu, var)
epsilon, F1 = select_threshold(y_val, p_val)

print('Best epsilon found using cross-validation: %e' % epsilon)
print('Best F1 on Cross Validation Set: %f' % F1)
    
# 单元测试
select_threshold_test(select_threshold)


**预期输出**：
<table>
  <tr>
    <td> <b>用交叉验证找到的最佳 epsilon：<b>  </td> 
    <td> 8.99e-05</td> 
   </tr>    
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.875 </td> 
  </tr>
</table>

现在我们将运行你的异常检测代码，并在图中圈出异常点（下面的图 3）。

<img src="图片/figure3.png" width="500" height="500">

> **图中文字（中文）**：横轴：延迟（Latency，ms） 图 3：分类出的异常点。

In [ ]:
# 找出训练集中的离群点
outliers = p < epsilon

# 可视化拟合效果
visualize_fit(X_train, mu, var)

# 在这些离群点周围画红圈
plt.plot(X_train[outliers, 0], X_train[outliers, 1], 'ro',
         markersize= 10,markerfacecolor='none', markeredgewidth=2)

<a name="2.4"></a>
### 2.4 高维数据集

现在，我们将在更真实、也更难的数据集上运行你实现的异常检测算法。

在这个数据集中，每个样本由 11 个特征描述，涵盖计算服务器的更多属性。

让我们先加载数据集。

- 下面显示的 `load_data()` 函数把数据加载到变量 `X_train_high`、`X_val_high` 和 `y_val_high` 中
    -  `_high` 用于把这些变量与上一部分使用的变量区分开
    - 我们将用 `X_train_high` 拟合高斯分布
    - 我们将把 `X_val_high` 和 `y_val_high` 用作交叉验证集，以选取阈值并判定样本是异常还是正常

In [ ]:
# 加载数据集
X_train_high, X_val_high, y_val_high = load_data_multi()

#### 查看变量的维度

让我们查看这些新变量的维度，以熟悉数据

In [ ]:
print ('The shape of X_train_high is:', X_train_high.shape)
print ('The shape of X_val_high is:', X_val_high.shape)
print ('The shape of y_val_high is: ', y_val_high.shape)

#### 异常检测

现在，让我们在这个新数据集上运行异常检测算法。

下面的代码将用你的代码
* 估计高斯参数（$\mu_i$ 和 $\sigma_i^2$）
* 计算用于估计高斯参数的训练数据 `X_train_high` 以及交叉验证集 `X_val_high` 的概率。
* 最后，它将用 `select_threshold` 找出最佳阈值 $\varepsilon$。

In [ ]:
# 对更大的数据集应用同样的步骤

# 估计高斯参数
mu_high, var_high = estimate_gaussian(X_train_high)

# 在训练集上评估概率
p_high = multivariate_gaussian(X_train_high, mu_high, var_high)

# 在交叉验证集上评估概率
p_val_high = multivariate_gaussian(X_val_high, mu_high, var_high)

# 寻找最佳阈值
epsilon_high, F1_high = select_threshold(y_val_high, p_val_high)

print('Best epsilon found using cross-validation: %e'% epsilon_high)
print('Best F1 on Cross Validation Set:  %f'% F1_high)
print('# Anomalies found: %d'% sum(p_high < epsilon_high))

**预期输出**：
<table>
  <tr>
    <td> <b>用交叉验证找到的最佳 epsilon：<b>  </td> 
    <td> 1.38e-18</td> 
   </tr>    
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.615385 </td> 
  </tr>
    <tr>
    <td> <b>发现的异常数量：<b>  </td>
     <td>  117 </td> 
  </tr>
</table>